In [ ]:
pip install pymupdf pandas pydantic openai sqlalchemy

In [ ]:
import fitz  # PyMuPDF

pdf= '/Users/faizaanm/Desktop/Contract Review Project/CONSTRUCTION SUBCONTRACT AGREEMENT(SAMPLE).pdf'

def extract_pdf_pages(pdf_path):
    doc = fitz.open(pdf_path)

    pages = []

    for page_number, page in enumerate(doc, start=1):
        text = page.get_text("text")

        pages.append({
            "page": page_number,
            "text": text
        })

    return pages

pages_e= extract_pdf_pages(pdf)

pages_e

In [ ]:
import re

def clean_text(text):
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()



In [ ]:
clean_pages = [] # Algorithm for going through lists with dicts in them derived for changing dict values learned. 

for page in pages_e:
    cleaned_page = page.copy()
    cleaned_page["text"] = clean_text(page["text"])
    clean_pages.append(cleaned_page)

clean_pages

In [ ]:
SECTION_PATTERN = re.compile(
    r"^(\d+)\.\s+([A-Z][A-Z\s/&-]+)$"
)

CLAUSE_PATTERN = re.compile(
    r"^(\d+\.\d+)\s+(.+)$"
)

In [ ]:
def parse_clauses(pages):
    clauses = []

    current_section = None
    current_clause = None

    for page in pages:
        page_number = page["page"]
        lines = page["text"].splitlines()

        for raw_line in lines:
            line = raw_line.strip()

            if not line:
                continue

            # Detect major section
            section_match = SECTION_PATTERN.match(line)

            if section_match:
                current_section = {
                    "section_number": section_match.group(1),
                    "section_title": section_match.group(2)
                }

                continue

            # Detect subsection/clause
            clause_match = CLAUSE_PATTERN.match(line)

            if clause_match:

                # Finish previous clause
                if current_clause:
                    current_clause["clause_text"] = (
                        "\n".join(current_clause["content"]).strip()
                    )

                    del current_clause["content"]
                    clauses.append(current_clause)

                current_clause = {
                    "clause_id": clause_match.group(1),
                    "clause_title": clause_match.group(2),
                    "section_number": (
                        current_section["section_number"]
                        if current_section else None
                    ),
                    "section_title": (
                        current_section["section_title"]
                        if current_section else None
                    ),
                    "page_start": page_number,
                    "page_end": page_number,
                    "content": []
                }

                continue

            # Add normal text
            if current_clause:
                current_clause["content"].append(line)
                current_clause["page_end"] = page_number

    # Add final clause
    if current_clause:
        current_clause["clause_text"] = (
            "\n".join(current_clause["content"]).strip()
        )

        del current_clause["content"]
        clauses.append(current_clause)

    return clauses

In [ ]:
clauses= parse_clauses(clean_pages)

clauses

In [ ]:
# Data Governance- Data tracking labels 

DOCUMENT_METADATA = {
    "document_id": "LCC-II-MECH-017",
    "document_version": "V1",
    "document_type": "Subcontract",
    "project_id": "LCC-II",
    "effective_date": "2026-09-01"
}

In [ ]:
for clause in clauses:
    clause.update(DOCUMENT_METADATA)

In [ ]:
clauses

In [ ]:
# DATA VALIDATION- check for issues 

def validate_clauses(clauses):
    issues = []

    seen = set()

    for clause in clauses:

        if clause["clause_id"] in seen:
            issues.append(
                f"Duplicate clause: {clause['clause_id']}"
            )

        seen.add(clause["clause_id"])

        if len(clause["clause_text"]) < 20:
            issues.append(
                f"Suspiciously short clause: {clause['clause_id']}"
            )

        if clause["page_start"] > clause["page_end"]:
            issues.append(
                f"Invalid page range: {clause['clause_id']}"
            )

    return issues


In [ ]:
issues=validate_clauses(clauses)

In [ ]:
print(issues)

In [ ]:
import pandas as pd

df = pd.DataFrame(clauses)

df.to_csv(
    "parsed_contract_clauses.csv",
    index=False
)

In [ ]:
from pydantic import BaseModel
from typing import Literal, Optional

class ClauseAnalysis(BaseModel):
    clause_id: str

    category: Literal[
        "Payment",
        "Schedule",
        "Change",
        "Delay",
        "Insurance",
        "Claims",
        "Termination",
        "Safety",
        "Warranty",
        "Data Governance",
        "Other"
    ]

    finding_type: Literal[
        "Risk",
        "Opportunity",
        "Obligation",
        "Neutral"
    ]

    severity: Literal[
        "Low",
        "Medium",
        "High",
        "Critical"
    ]

    finding: str

    recommended_action: str

    deadline_days: Optional[int] = None

    financial_impact: Optional[str] = None

    schedule_impact: Optional[str] = None

    confidence: float

In [ ]:
def build_prompt(clause):
    return f"""
You are reviewing a construction subcontract.

Analyze ONLY the clause provided below.

Do not infer obligations that are not explicitly supported
by the text.

Document ID: {clause['document_id']}
Document version: {clause['document_version']}
Clause: {clause['clause_id']}
Clause title: {clause['clause_title']}
Pages: {clause['page_start']}-{clause['page_end']}

CLAUSE TEXT:

{clause['clause_text']}

Identify:
- category
- whether this represents a risk, opportunity,
  obligation, or neutral provision
- severity
- concise finding
- recommended action
- explicit deadline, if present
- financial implications
- schedule implications
- confidence
"""

In [ ]:
%pip install openai pydantic

In [ ]:
from openai import OpenAI
from pydantic import BaseModel

In [ ]:
import sys
print(sys.executable)

In [ ]:
import os

# Set OPENAI_API_KEY in your environment before running this notebook.

In [ ]:
from openai import OpenAI

client = OpenAI()

In [ ]:
test_clause = clauses[0]

response = client.responses.parse(
    model="gpt-5.6-sol",
    input=build_prompt(test_clause),
    text_format=ClauseAnalysis
)

analysis = response.output_parsed
print(analysis)

In [ ]:
print(analysis.model_dump())

In [ ]:
result = analysis.model_dump()

result["clause_id"] = clause["clause_id"]

print(result)

In [ ]:
results = []

for clause in clauses:

    response = client.responses.parse(
        model="gpt-5.5",
        input=build_prompt(clause),
        text_format=ClauseAnalysis
    )

    result = response.output_parsed.model_dump()

    result["clause_id"] = clause["clause_id"]

    results.append(result)

In [ ]:
print(results)

In [ ]:
import json

with open("contract_risk_review(v2).json", "w") as f:
    json.dump(results, f, indent=4)